# 05. Adding Custom Function Tools to the IT Help Desk Agent

This script **re-versions** the same `"IT-HelpDesk-Agent"` created in `02_prompt_agent.py` — same
`agent_name`, so `client.agents.create_version(...)` publishes a **new version** (version 2, assuming this
runs after `02_prompt_agent.py`'s version 1) rather than creating a separate agent. What's new is the
`tools` list: three `FunctionTool` definitions describing, via hand-written JSON Schema, the same three
capabilities as `01_openai_agent.py`'s local tools (password reset, VPN troubleshooting, software install
guide) — but here, only the **schema** is registered with Foundry. The actual Python implementations
(`helpdesk_functions.py`) live entirely outside this script and are executed by the *caller*
(`07_helpdesk_client.py`), not by the Foundry service.

**Difficulty:** Intermediate


## Prerequisites

**pip3 packages:** `azure-ai-projects`, `azure-identity`, `python-dotenv` — already in root `requirements.txt`.
`FunctionTool` is exported from `azure.ai.projects.models`, same package.

**Azure resource(s) required:** The same Foundry project as `02_prompt_agent.py`. Note this script uses a
*different* model deployment name (`"gpt-4.1-mini"` in the original) than `02_prompt_agent.py`
(`"gpt-5.4"`) — both are placeholder deployment names from the course; use whatever deployment(s) actually
exist in your project.

**Environment variables** (root `.env`):
- `AZURE_AI_PROJECT_ENDPOINT`
- `AZURE_AI_MODEL_DEPLOYMENT` (can reuse the same variable as other scripts, or add a distinct one if you
  want this agent version on a different deployment)
- `AZURE_AI_AGENT_NAME` (optional, defaults to `"IT-HelpDesk-Agent"`)


## What You'll Learn

- How to declare a **custom `FunctionTool`** with an explicit JSON Schema (`parameters`) describing its
  arguments — unlike `01_openai_agent.py`, where the SDK inferred this from Python type hints
- What `strict=True` and `"additionalProperties": False` mean for tool-call argument validation
- That `FunctionTool` registers only a **contract** — Foundry never executes your function code; a client
  must do that (see `07_helpdesk_client.py` and the `helpdesk_functions.py` reference module)
- How `create_version` on an existing agent name adds a new version rather than overwriting or duplicating


### Setup and defining the function tool schemas

Each `FunctionTool(...)` mirrors one function in `helpdesk_functions.py`: a `name` the model will reference
when it wants to call it, a human-readable `description` the model uses to decide *when* to call it, and a
`parameters` JSON Schema describing its arguments. The first two tools take no arguments (`"properties": {}`,
`"required": []`); the third requires a `software_name` string. `strict=True` tells the service to enforce the
schema strictly (reject/repair malformed arguments) rather than loosely.

- **💡 Exam tip:** This is the same JSON-Schema-based function-calling contract used across OpenAI and Azure
  OpenAI function calling generally — AI-102 expects you to recognize `type`, `properties`, `required`, and
  `additionalProperties: false` as the standard shape of a tool's parameter schema.
- **🔄 Alternatives:** Bundling several such `FunctionTool` definitions into a single reusable `ToolSet` object
  (see `11_azure_ai_foundry/03_toolbox`) instead of a plain Python list, if you plan to reuse the same tool
  set across multiple agents.


In [ ]:
import os

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool
from azure.identity import DefaultAzureCredential

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT", "gpt-4.1-mini")
AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "IT-HelpDesk-Agent")

client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

tools = [
    FunctionTool(
        name="get_password_reset_steps",
        description="Get the company password reset steps.",
        parameters={
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
        strict=True,
    ),
    FunctionTool(
        name="get_vpn_troubleshooting_steps",
        description="Get troubleshooting steps for VPN connection issues.",
        parameters={
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
        strict=True,
    ),
    FunctionTool(
        name="get_software_install_guide",
        description="Get installation instructions for a supported software package.",
        parameters={
            "type": "object",
            "properties": {
                "software_name": {
                    "type": "string",
                    "description": "The software name, for example Slack, Zoom, or VS Code."
                }
            },
            "required": ["software_name"],
            "additionalProperties": False,
        },
        strict=True,
    ),
]


### Publishing the new agent version

Same `create_version` call shape as `02_prompt_agent.py`, now with `tools=tools` attached. Because
`agent_name` is unchanged (`"IT-HelpDesk-Agent"`), Foundry adds this as a new version of the existing agent —
this is why `07_helpdesk_client.py` later pins `AGENT_VERSION = "2"` specifically to target this
tool-equipped version rather than version 1's plain prompt agent.

- **💡 Exam tip:** Function tools declared here are **never invoked by the Foundry service itself** — when the
  model decides to call `get_vpn_troubleshooting_steps`, the Responses API returns a `function_call` output
  item and *stops*, waiting for the caller to execute the function and submit a `function_call_output`. This
  client-must-execute contract is the single most important thing to remember about `FunctionTool` for the
  exam.
- **🔄 Alternatives:** If the desired behavior could instead be served by a real external API, an
  **OpenAPI-based custom tool** pointed at that API (see `11. cloudxeus-func`/`12. cloudxeus-func1` later in
  this section) lets the *service* call the API directly via HTTP, rather than routing through your local
  Python process the way `FunctionTool` does.


In [ ]:
agent = client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=DEPLOYMENT_NAME,
        instructions=(
            "You are an IT support assistant for a company. "
            "Help users with password resets, VPN issues, and software installation. "
            "Give clear, step-by-step answers. "
            "If the question is outside IT support topics, politely say so."
        ),
        tools=tools
    )
)

print("Agent created:")
print(f"  ID      : {agent.id}")
print(f"  Name    : {agent.name}")
print(f"  Version : {agent.version}")


## Summary

Running this notebook publishes a new version of `IT-HelpDesk-Agent` that now advertises three custom
function tools via JSON Schema. Foundry itself still doesn't know how to run
`get_password_reset_steps`/`get_vpn_troubleshooting_steps`/`get_software_install_guide` — it only knows their
names, descriptions, and argument shapes. The real implementations live in `helpdesk_functions.py`, and the
loop that actually executes them and reports results back lives in `07_helpdesk_client.py`.


## Try It Yourself

1. **Easy:** Add a fourth `FunctionTool` (e.g. `get_printer_setup_steps`) with no arguments, matching a new
   function you'd add to `helpdesk_functions.py`.
2. **Intermediate:** Change `software_name`'s schema to an `enum` of `["slack", "zoom", "vscode"]` instead of a
   free-form string, and consider how that changes what arguments the model is allowed to send.
3. **Advanced:** Compare this version's `agent.version` value in your own Foundry project against the version
   printed by `02_prompt_agent.py` — confirm they differ and that both are versions of the same `agent.id`
   family (or look up `agent.name` to confirm identity).
